In [38]:
import pandas as pd
import mysql.connector

print(pd.__version__)
print(mysql.connector.__version__)

2.3.3
26.10.0


In [39]:
#Ruta al CSV final de HU5 (el notebook está en src, por eso subimos con ../)
SOURCE_PATH = "../data/HU5_medicamentos_diabetes_clusters.csv"

In [40]:
#guardamos los datos del csv
df = pd.read_csv(SOURCE_PATH, encoding="utf-8-sig")

In [41]:
print("Filas y columnas:", df.shape)
df.info()

Filas y columnas: (827, 30)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 827 entries, 0 to 826
Data columns (total 30 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   nregistro                        827 non-null    int64  
 1   nombre                           827 non-null    object 
 2   pactivos                         827 non-null    object 
 3   labtitular                       827 non-null    object 
 4   labcomercializador               827 non-null    object 
 5   cn                               827 non-null    int64  
 6   dosis                            827 non-null    object 
 7   forma_farmaceutica_simplificada  827 non-null    object 
 8   estado_aut                       827 non-null    object 
 9   comercializado                   827 non-null    int64  
 10  generico                         827 non-null    int64  
 11  afecta_conduccion                827 non-null    int64  

In [42]:
#vemos los tipos de nuestras columnas
print(df.dtypes)


nregistro                            int64
nombre                              object
pactivos                            object
labtitular                          object
labcomercializador                  object
cn                                   int64
dosis                               object
forma_farmaceutica_simplificada     object
estado_aut                          object
comercializado                       int64
generico                             int64
afecta_conduccion                    int64
biosimilar                           int64
num_registros_atc                    int64
num_principios_activos               int64
num_excipientes                      int64
vias_administracion                 object
palabras_seccion_4_4                 int64
numero_tablas                        int64
indicador_riesgo_severo              int64
estado                              object
precio_pvp_iva                     float64
tratamiento_larga_duracion           int64
tiene_revis

In [43]:
#Rangos de las columnas decimales para elegir las variables int y float(32,64 y eso)
print(df[["precio_pvp_iva", "dosis_por_ml", "dosis_numero_2", "dosis_numero"]].describe())

       precio_pvp_iva  dosis_por_ml  dosis_numero_2  dosis_numero
count      827.000000    827.000000      827.000000    827.000000
mean        20.299045      0.051995       97.910822     78.991389
std         23.180832      0.222152      288.265403    167.373226
min          0.950000      0.000000        0.000000      0.004000
25%          4.170000      0.000000        0.000000     10.000000
50%         11.180000      0.000000        0.000000     40.000000
75%         27.935000      0.000000        0.000000     75.000000
max        133.470000      1.000000     1000.000000   1000.000000


In [44]:
#Datos para conectar con MySQL
usuario = "root"                      
clave = input("Contraseña de MySQL: ") 

#Abrimos la conexión con la base de datos
conexion = mysql.connector.connect(
    host="localhost",
    user=usuario,
    password=clave,
    database="biodatapharma",
)

#Con el cursor mandamos las queries a MySQLb
cursor = conexion.cursor()

print("Conectado:", conexion.is_connected())

Conectado: True


In [45]:
cursor.execute("DROP TABLE IF EXISTS medicamentos_diabetes")
create_sql = """
CREATE TABLE IF NOT EXISTS medicamentos_diabetes (
    nregistro INT PRIMARY KEY,
    nombre VARCHAR(150),
    pactivos VARCHAR(64),
    labtitular VARCHAR(64),
    labcomercializador VARCHAR(64),
    cn INT,
    dosis VARCHAR(64),
    forma_farmaceutica_simplificada VARCHAR(64),
    estado_aut DATETIME,
    comercializado INT,
    generico INT,
    afecta_conduccion INT,
    biosimilar INT,
    num_registros_atc INT,
    num_principios_activos INT,
    num_excipientes INT,
    vias_administracion VARCHAR(64),
    palabras_seccion_4_4 INT,
    numero_tablas INT,
    indicador_riesgo_severo INT,
    estado VARCHAR(64),
    precio_pvp_iva DOUBLE,
    tratamiento_larga_duracion INT,
    tiene_revision INT,
    tiene_foto_materiales INT,
    dosis_por_ml INT,
    dosis_numero_2 DOUBLE,
    dosis_numero DOUBLE,
    cluster INT,
    perfil VARCHAR(64)
) 
"""
cursor.execute(create_sql)
print("Tabla creada")

Tabla creada


In [46]:
#Preparamos el INSERT con tantos %s como columnas (30).Los %s son huecos"que el conector rellena con cada valor 
huecos = ", ".join(["%s"] * len(df.columns)) #len da el numero de columnas,["%s"] es una lista con un solo elemento,multiplicandola por el len 
#hacemos que la lista pase a tener 30 veces ese elemento.Luego con el join cogemos esa lista de 30 elementos y la convertimos en un solo texto,
#que tendra 30 veces %s y separado por una , 
#con esto de debajo preparamos la instruccion para sql.Ahora mismo tenemos una tabla con 30 columnas vacias,y lo que queremos es insertar las filas.
#Por eso,huecos tiene 30 espacios(cada hueco de una fila),que rellenaremos con lo que guardemos en la variable filas
insert_sql = f"INSERT INTO medicamentos_diabetes VALUES ({huecos})"

#Convertimos el df en una lista de filas. astype(object) + tolist() convierte los números de numpy en números normales de Python,que es lo que entiende mysql.connector.
filas = df.astype(object).values.tolist()

#executemany inserta todas las filas de golpe
cursor.executemany(insert_sql, filas)

# commit = "guardar definitivamente". Sin commit, los datos no se quedan en la base.
conexion.commit()
print("Filas insertadas:", cursor.rowcount)

Filas insertadas: 827


In [47]:
# Creamos la vista con los 5 KPIs (es el mismo SQL que probamos en queries.txt).
# El SQL va dentro de un texto (comillas triples) y con cursor.execute se lo mandamos a MySQL
# NOTA SOBRE KPIs 4 y 5!!! : el enunciado usa el PR (precio de referencia), pero lo eliminamos en HU4 porque era igual al PVP y las fórmulas darían 0.
# Por eso los sustituimos por la diferencia con el mínimo del grupo y la desviación respecto a la media del grupo(los KPIs diferentes que hemos pensado)
# KPI 1: total_componentes = principios activos + excipientes
# KPI 2: densidad_ficha_tecnica = palabras de la seccion 4.4 entre numero de tablas (NULL si no hay tablas)
# KPI 3: ranking_precio_grupo = posicion por precio dentro de su forma farmaceutica (1 = el mas caro)
# KPI 4: diferencia_precio_minimo_grupo = precio menos el precio del mas barato de su grupo
# KPI 5: desviacion_precio_grupo = % que se aleja el precio de la media de su grupo
vista_sql = """
CREATE OR REPLACE VIEW kpis_medicamentos AS
SELECT
    nregistro,
    nombre,
    forma_farmaceutica_simplificada,
    precio_pvp_iva,
    num_principios_activos,
    num_excipientes,
    num_principios_activos + num_excipientes AS total_componentes,
    palabras_seccion_4_4,
    numero_tablas,
    palabras_seccion_4_4 / numero_tablas AS densidad_ficha_tecnica,
    ROW_NUMBER() OVER (
        PARTITION BY forma_farmaceutica_simplificada
        ORDER BY precio_pvp_iva DESC
    ) AS ranking_precio_grupo,
    ROUND(precio_pvp_iva - MIN(precio_pvp_iva) OVER (PARTITION BY forma_farmaceutica_simplificada), 2) AS diferencia_precio_minimo_grupo,
    ROUND(
        (precio_pvp_iva - AVG(precio_pvp_iva) OVER (PARTITION BY forma_farmaceutica_simplificada))
        / AVG(precio_pvp_iva) OVER (PARTITION BY forma_farmaceutica_simplificada) * 100,
        2
    ) AS desviacion_precio_grupo
FROM medicamentos_diabetes
"""

cursor.execute(vista_sql)
print("Vista creada")

Vista creada


In [48]:
#comprobamos lo que hemos creado 
cursor.execute("SELECT COUNT(*) FROM kpis_medicamentos")
print(cursor.fetchone())

(827,)
